<a href="https://colab.research.google.com/github/Joaoplims/sna_roblox_kg/blob/main/Roblox_KG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install requests

## SETUP

In [15]:
from google.colab import userdata

API_KEY = userdata.get('roblox_oauth')
print( API_KEY )
HEADERS = {
    "x-api-key": API_KEY,
    "Content-Type": "application/json"
}

Bo3UKVecGk6QtPtNkA5tb2DIT+PFVGIxoNpqERPlnIdzQp2RZXlKaGJHY2lPaUpTVXpJMU5pSXNJbXRwWkNJNkluTnBaeTB5TURJeExUQTNMVEV6VkRFNE9qVXhPalE1V2lJc0luUjVjQ0k2SWtwWFZDSjkuZXlKaGRXUWlPaUpTYjJKc2IzaEpiblJsY201aGJDSXNJbWx6Y3lJNklrTnNiM1ZrUVhWMGFHVnVkR2xqWVhScGIyNVRaWEoyYVdObElpd2lZbUZ6WlVGd2FVdGxlU0k2SWtKdk0xVkxWbVZqUjJzMlVYUlFkRTVyUVRWMFlqSkVTVlFyVUVaV1IwbDRiMDV3Y1VWU1VHeHVTV1I2VVhBeVVpSXNJbTkzYm1WeVNXUWlPaUl4TURJME5USXhPRGMwTnlJc0ltVjRjQ0k2TVRjNE1UUTVORGsxTml3aWFXRjBJam94TnpneE5Ea3hNelUyTENKdVltWWlPakUzT0RFME9URXpOVFo5LkRGV0VWVVh2N0dYdk5oWTVwbmFseW5wSjhHRUkwbzBoNEhsVDhobVdybFM2ZzZMWTF1M2gwSEZnRDZNY3VuUDN5XzhORS1GRWFHd3pEeDlYTVBEUWJiemRvMW90UEx1Z2t5WXNOVmZrRUpmV1JDNUwwcUp3VGN4dmdUYU1heldDcTMwRGFtWlBFNXVrTF85OHN6ZU9MWG5XOXBRMGV4cUtZelMzdFNTX0ZuNUIwcm5kb2NNeGltc3c5bTBONmhrQVBtdnBxckppb28zVDNCblkweUpCRWllT2YzSUdoM1JlTWcwYllvdE02b1pIek9GS096OElfZ1A3R0lKUnl4UDBaVWNxU0NxZGNzUHloem5UODkxYUpSRm40WTBZWHR4SEpsLVJNYnpRTDhST2RRZkxjcnlUVHIzdjZXVXgtZDZqVHFDU3ZzLXBHcERna1dONmJQVjc5Zw==


## Utilidades

In [3]:
def save_df_to_csv(df, filename="data_export.csv"):
    """
    Salva um DataFrame do pandas em um arquivo CSV.
    """
    try:
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"✅ Arquivo '{filename}' salvo com sucesso!")
    except Exception as e:
        print(f"❌ Erro ao salvar o arquivo: {e}")

# Exemplo de uso com os dados processados anteriormente:
# save_df_to_csv(df_users, "usuarios_roblox.csv")

## Função de Chamada para a API

In [18]:
import requests
import time
import random

def make_roblox_request(url, method="GET", params=None, data=None, max_retries=5, auto_paginate=False):
    """
    Função genérica para realizar chamadas à API do Roblox com suporte a Rate Limiting e Paginação.
    """
    all_data = []
    current_params = params.copy() if params else {}

    while True:
        retry_count = 0
        success = False

        while retry_count <= max_retries:
            try:
                response = requests.request(
                    method=method,
                    url=url,
                    headers=HEADERS,
                    params=current_params,
                    json=data
                )

                if response.status_code == 200:
                    res_json = response.json()

                    # Lógica de detecção: se houver uma chave 'data' e for uma lista, tratamos como paginável
                    if isinstance(res_json, dict) and 'data' in res_json and isinstance(res_json['data'], list):
                        page_items = res_json['data']
                        all_data.extend(page_items)

                        next_cursor = res_json.get('nextPageCursor')
                        if auto_paginate and next_cursor:
                            current_params['cursor'] = next_cursor
                            print(f"📄 Página processada. Buscando próxima página...")
                            success = True
                            break
                        else:
                            print(f"✅ Sucesso [{method}]: {url} (Total itens: {len(all_data)})")
                            return {"data": all_data}
                    else:
                        # Se não for uma lista paginável, retorna o JSON bruto (caso da Cloud API v2)
                        print(f"✅ Sucesso [{method}]: {url} (Objeto individual)")
                        return res_json

                elif response.status_code == 429:
                    retry_count += 1
                    wait_time = int(response.headers.get("retry-after", (2 ** retry_count) + random.uniform(0, 1)))
                    print(f"⚠️ Rate Limited (429). Aguardando {wait_time}s...")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"❌ Erro {response.status_code}: {response.text}")
                    return None
            except Exception as e:
                print(f"❌ Erro na requisição: {e}")
                return None

        if not success: break
    return None

## Teste chamadas


In [14]:
target_uid = 947838656
user_api_url = f"https://apis.roblox.com/cloud/v2/users/{target_uid}"

print(f"📡 Consultando dados para o usuário: {target_uid}...")
user_data_response = make_roblox_request(user_api_url)

import json
if user_data_response:
    print("✅ Resposta recebida:")
    print(json.dumps(user_data_response, indent=4))
else:
    print("❌ Não foi possível obter os dados do usuário.")

📡 Consultando dados para o usuário: 947838656...
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/947838656 (Total itens: 0)
✅ Resposta recebida:
{
    "data": []
}


In [17]:
import requests
import json

# Teste direto sem a função make_roblox_request
test_uid = 947838656
test_url = f"https://apis.roblox.com/cloud/v2/users/{test_uid}"

print(f"📡 Teste Direto (requests): {test_url}")
test_response = requests.get(test_url, headers=HEADERS)

print(f"Status Code: {test_response.status_code}")
try:
    raw_json = test_response.json()
    print("Raw JSON Response:")
    print(json.dumps(raw_json, indent=4))
except Exception as e:
    print(f"Erro ao decodificar JSON: {e}")
    print(f"Conteúdo bruto: {test_response.text}")

📡 Teste Direto (requests): https://apis.roblox.com/cloud/v2/users/947838656
Status Code: 200
Raw JSON Response:
{
    "path": "users/947838656",
    "createTime": "2019-01-28T20:56:39.770Z",
    "id": "947838656",
    "name": "Scriptbloxian",
    "displayName": "Scriptbloxian",
    "about": "Lead Developer of Scriptbloxian Studios\nhttps://www.roblox.com/groups/4705120/Scriptbloxian-Studios",
    "locale": "en_us",
    "premium": true
}


## Encontrar ID Seed Group

In [5]:
import time
# Cache do id do grupo mais popular (Scriptbloxian Studios) para evitar chamadas excessiva
ID_SEED_GROUP = 0
# Buscar o Grupo 'Scriptbloxian Studios'
# Pesquisas indicam que é um dos grupos mais populares (https://roblox.fandom.com/pt-br/wiki/Scriptbloxian_Studios)
# Grupo de desenvolvedor de jogos famoso
group_search_url = "https://groups.roblox.com/v1/groups/search/lookup?groupName=Scriptbloxian%20Studios"
group_search_results = make_roblox_request(group_search_url)

if group_search_results and group_search_results.get('data'):
    group_info = group_search_results['data'][0]
    group_id = group_info['id']
    ID_SEED_GROUP = group_id
    print(f"🎯 Grupo Encontrado: {group_info['name']} (ID: {group_id})")
else:
    print("❌ Grupo não encontrado ou erro na busca.")

✅ Sucesso [GET]: https://groups.roblox.com/v1/groups/search/lookup?groupName=Scriptbloxian%20Studios (Total itens: 10)
🎯 Grupo Encontrado: Scriptbloxian Studios (ID: 4705120)


## Consulta de usuários

In [8]:
SAMPLE_SIZE = 25  # Parametrizável

# 2. Extrair membros do grupo semente
members_url = f"https://groups.roblox.com/v1/groups/{ID_SEED_GROUP}/users?sortOrder=Asc&limit={SAMPLE_SIZE}"
members_data = make_roblox_request(members_url, auto_paginate = False)

user_ids = []
if members_data and 'data' in members_data:
    user_ids = [member['user']['userId'] for member in members_data['data']]
    print(f"👥 Extraídos {len(user_ids)} usuários do grupo semente (ID: {ID_SEED_GROUP}).")
    print(f"Amostra de IDs: {user_ids}")
else:
    print("❌ Não foi possível extrair os membros do grupo.")

✅ Sucesso [GET]: https://groups.roblox.com/v1/groups/4705120/users?sortOrder=Asc&limit=25 (Total itens: 25)
👥 Extraídos 25 usuários do grupo semente (ID: 4705120).
Amostra de IDs: [947838656, 1027961385, 357829182, 244267273, 740396998, 731822913, 871184693, 262953699, 940534557, 970378427, 386824609, 747056849, 452079889, 519533783, 174791020, 363117394, 1041550312, 1038182669, 727787047, 557051148, 1023552809, 112492866, 363983769, 1014686408, 726703596]


### Processamento de Dados dos Usuários
Nesta etapa, consultamos a API de Usuários da Cloud API para obter detalhes como `createTime` e `locale`, calculando métricas derivadas.

In [19]:
from datetime import datetime, timezone
import pandas as pd
import time

processed_users = []
now = datetime.now(timezone.utc)

print(f"🔍 Processando {len(user_ids)} usuários...\n")

for uid in user_ids:
    url = f"https://apis.roblox.com/cloud/v2/users/{uid}"
    # A função corrigida agora retorna o objeto diretamente para a Cloud API v2
    user_data = make_roblox_request(url, auto_paginate=False)

    if user_data and user_data.get('createTime'):
        create_time_str = user_data['createTime']
        create_time_dt = datetime.fromisoformat(create_time_str.replace('Z', '+00:00'))

        delta = now - create_time_dt
        age = delta.days / 365.25

        processed_users.append({
            "user_id": user_data.get('id'),
            "display_name": user_data.get('displayName'),
            "created_at": create_time_str,
            "age_years": round(age, 2),
            "level": "Veterano" if age >= 10 else "Intermediário" if age >= 4 else "Iniciante"
        })
    else:
        print(f"⚠️ Falha ao obter detalhes do usuário {uid}")

    time.sleep(0.2)

df_users = pd.DataFrame(processed_users)
if not df_users.empty:
    display(df_users.head(10))
else:
    print("❌ Nenhum dado processado.")

🔍 Processando 25 usuários...

✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/947838656 (Objeto individual)
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/1027961385 (Objeto individual)
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/357829182 (Objeto individual)
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/244267273 (Objeto individual)
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/740396998 (Objeto individual)
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/731822913 (Objeto individual)
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/871184693 (Objeto individual)
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/262953699 (Objeto individual)
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/940534557 (Objeto individual)
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/970378427 (Objeto individual)
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/386824609 (Objeto individual)
✅ Sucesso [GET]: https:

,user_id,display_name,created_at,age_years,level
0,947838656,Scriptbloxian,2019-01-28T20:56:39.770Z,7.55,Intermediário
1,1027961385,Iamsmarterthanyahya3,2019-04-04T20:14:50.953Z,7.37,Intermediário
2,357829182,FallenGh,2017-08-02T17:39:17.880Z,9.04,Intermediário
3,244267273,kidsgivemesloppy,2017-02-20T05:17:52.727Z,9.49,Intermediário
4,740396998,Lloyd,2018-08-30T19:34:01.370Z,7.96,Intermediário
5,731822913,Kirill_pro300,2018-08-26T16:09:21.050Z,7.97,Intermediário
6,871184693,hunika201,2018-11-24T09:20:23.383Z,7.73,Intermediário
7,262953699,kevin2598,2017-03-14T15:33:27.687Z,9.42,Intermediário
8,940534557,furiouse_2,2019-01-23T01:56:45.160Z,7.56,Intermediário
9,970378427,Mid_Bacon,2019-02-16T16:54:47.510Z,7.50,Intermediário


### Exportação de Dados
Salvando o conjunto de dados processado em um arquivo CSV para uso externo.

In [ ]:
# Exportando os usuários processados
save_df_to_csv(df_users, "usuarios_roblox_processados.csv")

✅ Arquivo 'usuarios_roblox_processados.csv' salvo com sucesso!


## First Hop

### Mapeamento de Afiliações (First Hop)
Nesta etapa, buscamos todos os grupos de cada um dos usuários da nossa amostra inicial para descobrir as conexões entre eles.

In [22]:
graph_edges = []

print(f"🕸️ Iniciando mapeamento de afiliações para {len(user_ids)} usuários...")

for uid in user_ids:
    # Endpoint da API V1 para listar grupos/cargos do usuário
    affiliations_url = f"https://groups.roblox.com/v1/users/{uid}/groups/roles"

    # Como este endpoint retorna uma lista na chave 'data', nossa função make_roblox_request funcionará bem
    response = make_roblox_request(affiliations_url, auto_paginate=False)

    if response and 'data' in response:
        groups = response['data']
        for item in groups:
            group_info = item.get('group', {})
            role_info = item.get('role', {})

            # Registramos a aresta: Origem (User) -> Relação (Member) -> Destino (Group)
            edge = {
                "source_id": uid,
                "source_type": "User",
                "target_id": group_info.get('id'),
                "target_name": group_info.get('name'),
                "target_type": "Group",
                "role_name": role_info.get('name'),
                "role_rank": role_info.get('rank')
            }
            graph_edges.append(edge)

    # Pequeno delay para respeitar a API
    time.sleep(0.2)



# Criando um DataFrame para visualizar as conexões
df_graph = pd.DataFrame(graph_edges)
print(f"✅ Mapeamento concluído! Encontradas {len(df_graph)} conexões.")
display(df_graph.head(10))

🕸️ Iniciando mapeamento de afiliações para 25 usuários...
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/947838656/groups/roles (Total itens: 9)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/1027961385/groups/roles (Total itens: 1)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/357829182/groups/roles (Total itens: 28)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/244267273/groups/roles (Total itens: 14)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/740396998/groups/roles (Total itens: 27)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/731822913/groups/roles (Total itens: 4)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/871184693/groups/roles (Total itens: 4)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/262953699/groups/roles (Total itens: 5)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/940534557/groups/roles (Total itens: 3)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/970378427/groups/roles (Total itens: 6)
✅ Sucesso [GET]: h

KeyError: 'from_user_id'

In [24]:
# Exibindo uma amostra das arestas criadas
if graph_edges:
    print("Amostra das conexões:")
    for edge in graph_edges[:35]:
        print(f"Usuário [{edge['source_id']}] participando do Grupo [{edge['target_name']}] como '{edge['role_name']}' (Rank: {edge['role_rank']})")
else:
    print("❌ Nenhuma aresta encontrada")

Amostra das conexões:
Usuário [947838656] participando do Grupo [Mischief Mechanics] como 'Owner' (Rank: 255)
Usuário [947838656] participando do Grupo [XP Games.] como '[D] Developer' (Rank: 253)
Usuário [947838656] participando do Grupo [Titan Trainers] como 'Admin' (Rank: 254)
Usuário [947838656] participando do Grupo [Hero Rivals Arena] como 'Owner' (Rank: 255)
Usuário [947838656] participando do Grupo [Super Slide] como 'Owner' (Rank: 255)
Usuário [947838656] participando do Grupo [Shinoro Studios] como 'Owner' (Rank: 255)
Usuário [947838656] participando do Grupo [Magnetworks] como 'Owner' (Rank: 255)
Usuário [947838656] participando do Grupo [Scriptbloxian Creation Group] como 'Owner' (Rank: 255)
Usuário [947838656] participando do Grupo [Scriptbloxian Studios] como 'Lead Developer' (Rank: 255)
Usuário [1027961385] participando do Grupo [Scriptbloxian Studios] como 'Legend' (Rank: 1)
Usuário [357829182] participando do Grupo [Krabby Krew] como 'Anchovies' (Rank: 1)
Usuário [3578

In [21]:
# Salvar o progresso do grafo
save_df_to_csv(df_graph, "conexoes_roblox_first_hop.csv")

✅ Arquivo 'conexoes_roblox_first_hop.csv' salvo com sucesso!
